# Smart House Price Prediction

## Notebook 3: Model Training

This notebook trains multiple machine learning regression models using the preprocessed housing dataset.

The objectives are to:

- Load the processed dataset.
- Split the data into training and testing sets.
- Train multiple regression models.
- Evaluate each model using standard regression metrics.
- Compare model performance.
- Save the best-performing model for deployment.

In [53]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import joblib
import os

# Display plots inside notebook
%matplotlib inline


# 1. Configure Project Directories

To keep the notebook portable and independent of absolute file paths, all important project directories are configured using `pathlib`.

These paths will be used throughout the notebook to load the processed dataset and save trained models.

In [29]:
# ============================================================
# Configure Project Directories
# ============================================================

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
IMAGES_DIR = PROJECT_ROOT / "images"

print("Project directories configured successfully!")

print(f"\nProject Root : {PROJECT_ROOT}")


Project directories configured successfully!

Project Root : C:\Users\HP\AI Projects\Smart-House-Price-Prediction


# 2. Load the Processed Dataset

The fully preprocessed dataset generated in Notebook 2 is loaded for model training.

This dataset already includes:

- Missing value handling
- Ordinal encoding
- One-Hot encoding
- Feature scaling

No additional preprocessing is required before training the models.

In [30]:
# ============================================================
# Load Processed Dataset
# ============================================================

processed_df = pd.read_csv(PROCESSED_DATA_DIR / "processed_train.csv")

print("Processed dataset loaded successfully!")

print(f"\nDataset Shape: {processed_df.shape}")

processed_df.head()


Processed dataset loaded successfully!

Dataset Shape: (1460, 211)


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SalePrice
0,-1.730865,0.073375,-0.220875,-0.207142,0.651479,-0.517200,1.050994,0.878668,0.514104,1.052302,...,-0.058621,-0.301962,-0.045376,0.390293,-0.052414,-0.091035,-0.117851,0.467651,-0.305995,208500
1,-1.728492,-0.872563,0.460320,-0.091886,-0.071836,2.179628,0.156734,-0.429577,-0.570750,-0.689604,...,-0.058621,-0.301962,-0.045376,0.390293,-0.052414,-0.091035,-0.117851,0.467651,-0.305995,181500
2,-1.726120,0.073375,-0.084636,0.073480,0.651479,-0.517200,0.984752,0.830215,0.325915,1.052302,...,-0.058621,-0.301962,-0.045376,0.390293,-0.052414,-0.091035,-0.117851,0.467651,-0.305995,223500
3,-1.723747,0.309859,-0.447940,-0.096897,0.651479,-0.517200,-1.863632,-0.720298,-0.570750,-0.689604,...,-0.058621,-0.301962,-0.045376,0.390293,-0.052414,-0.091035,-0.117851,-2.138345,-0.305995,140000
4,-1.721374,0.073375,0.641972,0.375148,1.374795,-0.517200,0.951632,0.733308,1.366489,1.052302,...,-0.058621,-0.301962,-0.045376,0.390293,-0.052414,-0.091035,-0.117851,0.467651,-0.305995,250000


# 3. Separate Features and Target Variable

The processed dataset is divided into:

- **Features (`X`)**: All independent variables used for prediction.
- **Target (`y`)**: The house sale price (`SalePrice`).

The target variable is excluded from the feature matrix before model training.

In [31]:
# ============================================================
# Separate Features and Target
# ============================================================

X = processed_df.drop(columns=["SalePrice"])
y = processed_df["SalePrice"]

print("Feature and target variables created successfully!")

print(f"\nFeature Matrix Shape : {X.shape}")
print(f"Target Vector Shape  : {y.shape}")


Feature and target variables created successfully!

Feature Matrix Shape : (1460, 210)
Target Vector Shape  : (1460,)


# 4. Remove Non-Predictive Features

Some columns act only as identifiers and do not contain meaningful information for predicting house prices.

The `Id` column uniquely identifies each house but has no predictive relationship with the target variable.

Removing such features helps prevent the model from learning meaningless patterns and improves generalization.

In [32]:
# ============================================================
# Remove Non-Predictive Features
# ============================================================

if "Id" in X.columns:
    X = X.drop(columns=["Id"])
    print("'Id' column removed successfully.")
else:
    print("'Id' column not found.")

print(f"\nUpdated Feature Matrix Shape: {X.shape}")

# ============================================================
# Save Feature Columns (Must Match Training Data)
# ============================================================

feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    PROJECT_ROOT / "models" / "feature_columns.pkl"
)

print("\nFeature columns regenerated successfully!")
print(f"Total features: {len(feature_columns)}")
print(f"Contains Id: {'Id' in feature_columns}")


'Id' column removed successfully.

Updated Feature Matrix Shape: (1460, 209)

Feature columns regenerated successfully!
Total features: 209
Contains Id: False


In [33]:
loaded_columns = joblib.load(PROJECT_ROOT / "models" / "feature_columns.pkl")

print("Feature Columns:", len(loaded_columns))
print("Model Features:", len(best_model.feature_names_in_))
print("Perfect Match:", loaded_columns == list(best_model.feature_names_in_))


Feature Columns: 209
Model Features: 209
Perfect Match: True


# 5. Split the Dataset into Training and Testing Sets

To evaluate how well the models generalize to unseen data, the dataset is divided into:

- **Training Set (80%)**: Used to train the machine learning models.
- **Testing Set (20%)**: Used only for evaluating model performance.

A fixed `random_state` is used to ensure reproducibility of the results.

In [34]:
# ============================================================
# Train-Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train-test split completed successfully!")

print(f"\nTraining Features : {X_train.shape}")
print(f"Testing Features  : {X_test.shape}")

print(f"\nTraining Target   : {y_train.shape}")
print(f"Testing Target    : {y_test.shape}")


Train-test split completed successfully!

Training Features : (1168, 209)
Testing Features  : (292, 209)

Training Target   : (1168,)
Testing Target    : (292,)


# 6. Create a Generic Model Training Function

Instead of writing separate code for each regression model, a reusable training function is created.

The function will:

1. Train the model.
2. Generate predictions on the test set.
3. Calculate regression metrics.
4. Return the trained model and its evaluation results.

This approach makes the notebook cleaner, easier to maintain, and simplifies adding new models in the future.

# ENHANCED

# 6. Unified Model Training and Evaluation Pipeline

To ensure that every regression model is trained and evaluated consistently, a reusable evaluation pipeline is implemented.

The pipeline performs:

- Model training
- Prediction
- Regression metric calculation
- Cross-validation
- Training time measurement
- Prediction time measurement

Using a single evaluation function avoids duplicate code and ensures that all models are compared fairly using identical evaluation criteria.

In [35]:
# # ============================================================
# # Generic Model Training Function
# # ============================================================

# def train_and_evaluate(model, model_name):
#     """
#     Train a regression model and evaluate its performance.

#     Parameters
#     ----------
#     model : sklearn estimator
#         Regression model to train.
#     model_name : str
#         Name of the regression model.

#     Returns
#     -------
#     dict
#         Dictionary containing evaluation metrics.
#     """

#     # Train the model
#     model.fit(X_train, y_train)

#     # Predict on test data
#     y_pred = model.predict(X_test)

#     # Calculate metrics
#     r2 = r2_score(y_test, y_pred)
#     mae = mean_absolute_error(y_test, y_pred)
#     mse = mean_squared_error(y_test, y_pred)
#     rmse = np.sqrt(mse)

#     return {
#         "Model": model_name,
#         "R2 Score": r2,
#         "MAE": mae,
#         "MSE": mse,
#         "RMSE": rmse,
#         "Model Object": model
#     }

# ============================================================
# Enhanced Model Training Function
# ============================================================

def evaluate_model(
    model,
    model_name,
    X_train,
    X_test,
    y_train,
    y_test,
    cv=5
):
    """
    Train and evaluate a regression model.

    Parameters
    ----------
    model : sklearn estimator
        Model to be trained.

    model_name : str
        Name of the model.

    Returns
    -------
    dict
        Evaluation metrics and trained model.
    """

    # ---------------------------
    # Model Training
    # ---------------------------

    train_start = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - train_start

    # ---------------------------
    # Prediction
    # ---------------------------

    predict_start = time.time()

    predictions = model.predict(X_test)

    prediction_time = time.time() - predict_start

    # ---------------------------
    # Evaluation Metrics
    # ---------------------------

    r2 = r2_score(y_test, predictions)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    # ---------------------------
    # Cross Validation
    # ---------------------------

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=KFold(
            n_splits=cv,
            shuffle=True,
            random_state=42
        ),
        scoring="r2",
        n_jobs=-1
    )

    return {

        "Model": model_name,

        "R2 Score": r2,

        "CV Mean": cv_scores.mean(),

        "CV Std": cv_scores.std(),

        "MAE": mae,

        "MSE": mse,

        "RMSE": rmse,

        "Training Time (s)": training_time,

        "Prediction Time (s)": prediction_time,

        "Model Object": model
    }

# ============================================================
# Display Model Results
# ============================================================

def display_results(results):
    """
    Display evaluation metrics in a consistent format.
    """

    print(f"{results['Model']} trained successfully!\n")

    for metric, value in results.items():

        if metric == "Model Object":
            continue

        if isinstance(value, float):
            print(f"{metric:<22}: {value:.4f}")
        else:
            print(f"{metric:<22}: {value}")



# 7. Train Linear Regression Model

Linear Regression serves as the baseline model for this project.

It assumes a linear relationship between the input features and the target variable.

The performance of more advanced models will later be compared against this baseline.

# ENHANCED

# 7. Train and Evaluate Linear Regression

Linear Regression serves as the baseline model for the regression task. It establishes a reference point against which more sophisticated machine learning algorithms can be compared.

The enhanced evaluation pipeline reports:

- Test R² Score
- Cross-Validation Mean R²
- Cross-Validation Standard Deviation
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Training Time
- Prediction Time

In [36]:
# ============================================================
# Train Linear Regression
# ============================================================

# linear_regression_results = train_and_evaluate(
#     LinearRegression(),
#     "Linear Regression"
# )

# print("Linear Regression trained successfully!\n")

# for metric, value in linear_regression_results.items():
#     if metric != "Model Object":
#         if isinstance(value, float):
#             print(f"{metric:<12}: {value:.4f}")
#         else:
#             print(f"{metric:<12}: {value}")

#------------------ENHANCED----------------------

# ============================================================
# Linear Regression
# ============================================================

linear_regression_results = evaluate_model(
    model=LinearRegression(),
    model_name="Linear Regression",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(linear_regression_results)


Linear Regression trained successfully!

Model                 : Linear Regression
R2 Score              : 0.6549
CV Mean               : 0.7792
CV Std                : 0.0839
MAE                   : 22117.1947
MSE                   : 2647341156.5360
RMSE                  : 51452.3193
Training Time (s)     : 0.1950
Prediction Time (s)   : 0.0042


# 8. Train Ridge Regression Model

Ridge Regression is an extension of Linear Regression that incorporates **L2 regularization**.

By adding a penalty term to the loss function, Ridge Regression helps reduce overfitting and improves the model's ability to generalize, particularly when dealing with highly correlated features.

# ENHANCED

# 8. Train and Evaluate Ridge Regression

Ridge Regression extends Linear Regression by introducing L2 regularization, which penalizes excessively large coefficients.

Regularization helps reduce overfitting and improves the model's ability to generalize, particularly when dealing with correlated features.

The enhanced evaluation pipeline measures prediction accuracy, cross-validation performance, computational efficiency, and regression error metrics.

In [37]:
# ============================================================
# Train Ridge Regression
# ============================================================

# ridge_results = train_and_evaluate(
#     Ridge(alpha=1.0),
#     "Ridge Regression"
# )

# print("Ridge Regression trained successfully!\n")

# for metric, value in ridge_results.items():
#     if metric != "Model Object":
#         if isinstance(value, float):
#             print(f"{metric:<12}: {value:.4f}")
#         else:
#             print(f"{metric:<12}: {value}")

# ------------------ENHANCED----------------------

# ============================================================
# Ridge Regression
# ============================================================

ridge_results = evaluate_model(
    model=Ridge(alpha=1.0),
    model_name="Ridge Regression",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(ridge_results)


Ridge Regression trained successfully!

Model                 : Ridge Regression
R2 Score              : 0.6894
CV Mean               : 0.7843
CV Std                : 0.0815
MAE                   : 21773.1038
MSE                   : 2382283775.7948
RMSE                  : 48808.6445
Training Time (s)     : 0.0236
Prediction Time (s)   : 0.0055


# 9. Train Lasso Regression Model

Lasso Regression extends Linear Regression by incorporating **L1 regularization**.

Unlike Ridge Regression, Lasso can shrink some feature coefficients exactly to zero, effectively performing automatic feature selection. This makes it useful for reducing model complexity and identifying the most influential predictors.

# ENHANCED

# 9. Train and Evaluate Lasso Regression

Lasso Regression extends Linear Regression by applying L1 regularization, encouraging sparse model coefficients.

This regularization technique can perform implicit feature selection by shrinking less important coefficients toward zero, making the resulting model more interpretable.

The enhanced evaluation pipeline measures predictive performance, computational efficiency, and generalization capability using cross-validation.

In [38]:
# ============================================================
# Train Lasso Regression
# ============================================================

# lasso_results = train_and_evaluate(
#     Lasso(alpha=0.001, max_iter=10000),
#     "Lasso Regression"
# )

# print("Lasso Regression trained successfully!\n")

# for metric, value in lasso_results.items():
#     if metric != "Model Object":
#         if isinstance(value, float):
#             print(f"{metric:<12}: {value:.4f}")
#         else:
#             print(f"{metric:<12}: {value}")

#---------------------ENHANCED----------------------

# ============================================================
# Lasso Regression
# ============================================================

lasso_results = evaluate_model(
    model=Lasso(alpha=0.01, max_iter=50000, tol=1e-3, random_state=42),
    model_name="Lasso Regression",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(lasso_results)


Lasso Regression trained successfully!

Model                 : Lasso Regression
R2 Score              : 0.6549
CV Mean               : 0.7818
CV Std                : 0.0797
MAE                   : 22116.8406
MSE                   : 2647292976.4203
RMSE                  : 51451.8510
Training Time (s)     : 2.2717
Prediction Time (s)   : 0.0020


# 10. Train Decision Tree Regressor

Decision Tree Regression is a non-linear supervised learning algorithm that predicts the target value by recursively partitioning the feature space.

Unlike linear models, Decision Trees can capture complex relationships and interactions between features without requiring feature scaling.

The model performance depends on parameters such as tree depth and minimum samples required for splitting.

# ENHANCED

# 10. Train and Evaluate Decision Tree Regressor

Decision Tree Regression captures complex non-linear relationships by recursively partitioning the feature space into homogeneous regions.

Unlike linear models, decision trees do not assume a linear relationship between the input features and the target variable. However, they are susceptible to overfitting if appropriate hyperparameters are not applied.

The model is evaluated using the unified evaluation pipeline to assess predictive performance, computational efficiency, and generalization capability.

In [39]:
# ============================================================
# Train Decision Tree Regressor
# ============================================================

# decision_tree_results = train_and_evaluate(
#     DecisionTreeRegressor(
#         random_state=42,
#         max_depth=15,
#         min_samples_split=5,
#         min_samples_leaf=2
#     ),
#     "Decision Tree"
# )

# print("Decision Tree trained successfully!\n")

# for metric, value in decision_tree_results.items():
#     if metric != "Model Object":
#         if isinstance(value, float):
#             print(f"{metric:<12}: {value:.4f}")
#         else:
#             print(f"{metric:<12}: {value}")

#---------------------ENHANCED----------------------

# ============================================================
# Decision Tree Regressor
# ============================================================

decision_tree = DecisionTreeRegressor(
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

decision_tree_results = evaluate_model(
    model=decision_tree,
    model_name="Decision Tree",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(decision_tree_results)


Decision Tree trained successfully!

Model                 : Decision Tree
R2 Score              : 0.8030
CV Mean               : 0.6783
CV Std                : 0.1033
MAE                   : 25484.3651
MSE                   : 1511093884.8381
RMSE                  : 38872.7911
Training Time (s)     : 0.0513
Prediction Time (s)   : 0.0043


# 11. Train Random Forest Regressor

Random Forest Regression is an ensemble learning algorithm that combines the predictions of multiple Decision Trees.

Each tree is trained on a random subset of the training data and features, reducing overfitting and improving prediction accuracy.

Compared to a single Decision Tree, Random Forest generally provides:
- Higher predictive performance
- Better generalization
- Greater robustness to noise and outliers

# ENHANCED

# 11. Train and Evaluate Random Forest Regressor

Random Forest Regression is an ensemble learning algorithm that combines multiple decision trees using bootstrap aggregation (bagging).

By averaging the predictions of many independently trained trees, Random Forest reduces variance, improves generalization, and is generally more robust to overfitting than a single decision tree.

The model is evaluated using the unified evaluation pipeline, which reports predictive accuracy, cross-validation performance, computational efficiency, and regression error metrics.

In [40]:
# ============================================================
# Train Random Forest Regressor
# ============================================================

# random_forest_results = train_and_evaluate(
#     RandomForestRegressor(
#         n_estimators=300,
#         max_depth=20,
#         min_samples_split=2,
#         min_samples_leaf=1,
#         random_state=42,
#         n_jobs=-1
#     ),
#     "Random Forest"
# )

# print("Random Forest trained successfully!\n")

# for metric, value in random_forest_results.items():
#     if metric != "Model Object":
#         if isinstance(value, float):
#             print(f"{metric:<12}: {value:.4f}")
#         else:
#             print(f"{metric:<12}: {value}")

#---------------------ENHANCED----------------------

# ============================================================
# Random Forest Regressor
# ============================================================

random_forest = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

random_forest_results = evaluate_model(
    model=random_forest,
    model_name="Random Forest",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(random_forest_results)


Random Forest trained successfully!

Model                 : Random Forest
R2 Score              : 0.8940
CV Mean               : 0.8456
CV Std                : 0.0402
MAE                   : 17261.0636
MSE                   : 813231410.3797
RMSE                  : 28517.2125
Training Time (s)     : 1.6633
Prediction Time (s)   : 0.0779


# 12. Train Gradient Boosting Regressor

Gradient Boosting Regression is an ensemble learning technique that builds a sequence of weak learners (typically shallow decision trees), where each new model is trained to correct the errors made by the previous ones.

Unlike Random Forest, which trains trees independently, Gradient Boosting builds trees sequentially, allowing it to capture complex patterns in the data more effectively.

Gradient Boosting is widely used for structured/tabular datasets due to its strong predictive performance.

# ENHANCED

# 12. Train and Evaluate Gradient Boosting Regressor

Gradient Boosting Regression is an ensemble learning technique that builds a sequence of weak decision trees, where each new tree focuses on correcting the errors made by the previous ensemble.

Unlike Random Forest, which trains trees independently, Gradient Boosting learns sequentially, allowing it to model complex relationships with high predictive accuracy.

The model is evaluated using the unified evaluation pipeline, including regression metrics, cross-validation performance, computational efficiency, and generalization capability.

In [41]:
# ============================================================
# Train Gradient Boosting Regressor
# ============================================================

# gradient_boosting_results = train_and_evaluate(
#     GradientBoostingRegressor(
#         n_estimators=300,
#         learning_rate=0.05,
#         max_depth=3,
#         subsample=0.8,
#         random_state=42
#     ),
#     "Gradient Boosting"
# )

# print("Gradient Boosting trained successfully!\n")

# for metric, value in gradient_boosting_results.items():
#     if metric != "Model Object":
#         if isinstance(value, float):
#             print(f"{metric:<12}: {value:.4f}")
#         else:
#             print(f"{metric:<12}: {value}")

#---------------------ENHANCED----------------------

# ============================================================
# Gradient Boosting Regressor
# ============================================================

gradient_boosting = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gradient_boosting_results = evaluate_model(
    model=gradient_boosting,
    model_name="Gradient Boosting",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(gradient_boosting_results)


Gradient Boosting trained successfully!

Model                 : Gradient Boosting
R2 Score              : 0.9084
CV Mean               : 0.8791
CV Std                : 0.0465
MAE                   : 16081.3019
MSE                   : 702663788.8606
RMSE                  : 26507.8062
Training Time (s)     : 2.9336
Prediction Time (s)   : 0.0055


# 13. Train and Evaluate Extra Trees Regressor

Extra Trees (Extremely Randomized Trees) is an ensemble learning algorithm that extends the Random Forest approach by introducing additional randomness during tree construction.

Unlike Random Forest, Extra Trees selects split thresholds randomly rather than searching for the optimal split. This often reduces variance, speeds up training, and can improve generalization performance.

The model is evaluated using the same unified evaluation pipeline to enable a fair comparison with all previously trained regression models.

In [42]:
# ============================================================
# Extra Trees Regressor
# ============================================================

extra_trees = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

extra_trees_results = evaluate_model(
    model=extra_trees,
    model_name="Extra Trees",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(extra_trees_results)


Extra Trees trained successfully!

Model                 : Extra Trees
R2 Score              : 0.8925
CV Mean               : 0.8490
CV Std                : 0.0345
MAE                   : 17263.9090
MSE                   : 824192239.7035
RMSE                  : 28708.7485
Training Time (s)     : 2.8355
Prediction Time (s)   : 0.1077


# 14. Train and Evaluate AdaBoost Regressor

AdaBoost (Adaptive Boosting) is an ensemble learning algorithm that combines multiple weak learners to form a stronger predictive model.

Unlike bagging-based methods, AdaBoost trains models sequentially, where each subsequent learner focuses on correcting the errors made by previous learners. This adaptive learning strategy can improve prediction accuracy on complex datasets.

The model is evaluated using the same unified evaluation pipeline to ensure a consistent comparison with all other regression algorithms.

In [43]:
# ============================================================
# AdaBoost Regressor
# ============================================================

ada_boost = AdaBoostRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

ada_boost_results = evaluate_model(
    model=ada_boost,
    model_name="AdaBoost",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

display_results(ada_boost_results)


AdaBoost trained successfully!

Model                 : AdaBoost
R2 Score              : 0.8503
CV Mean               : 0.7764
CV Std                : 0.0376
MAE                   : 22892.5488
MSE                   : 1148329331.7375
RMSE                  : 33887.0083
Training Time (s)     : 5.7183
Prediction Time (s)   : 0.0720


# 15. Hyperparameter Optimization using RandomizedSearchCV

Hyperparameter optimization is performed on the top-performing ensemble models to improve predictive performance and generalization capability.

RandomizedSearchCV is selected instead of GridSearchCV because it efficiently explores the hyperparameter space while requiring significantly less computational time.

Each model is evaluated using 5-fold cross-validation, and the best hyperparameter configuration is retained for subsequent comparison.

In [44]:
# ============================================================
# Hyperparameter Search Spaces
# ============================================================

random_forest_params = {
    "n_estimators": [200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

gradient_boosting_params = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0]
}

extra_trees_params = {
    "n_estimators": [200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}


## 15.1 Hyperparameter Optimization Function

To avoid repetitive code, a reusable helper function is defined for performing hyperparameter optimization using RandomizedSearchCV.

The function performs the following steps:

- Executes randomized hyperparameter search using 5-fold cross-validation.
- Identifies the best parameter combination.
- Retrains the optimized model.
- Evaluates the optimized model using the unified evaluation framework.
- Returns both the evaluation results and the optimal hyperparameters.

This modular implementation improves readability, maintainability, and scalability of the training pipeline.

In [45]:
# ============================================================
# Hyperparameter Optimization Function
# ============================================================

def tune_model(
    model,
    param_distributions,
    model_name,
    X_train,
    X_test,
    y_train,
    y_test,
    n_iter=20,
    cv=5
):
    """
    Perform RandomizedSearchCV and evaluate the optimized model.
    """

    print("=" * 70)
    print(f"Tuning {model_name}...")
    print("=" * 70)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="r2",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_

    print("\nBest Parameters:")
    print(random_search.best_params_)

    print(f"\nBest Cross Validation Score: {random_search.best_score_:.4f}")

    results = evaluate_model(
        model=best_model,
        model_name=f"{model_name} (Tuned)",
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    )

    display_results(results)

    return best_model, results, random_search.best_params_


## 15.2 Hyperparameter Optimization - Random Forest

Random Forest is one of the strongest baseline models in this study. Hyperparameter optimization is performed using RandomizedSearchCV to identify the optimal combination of parameters that maximizes predictive performance while maintaining good generalization.

The optimized model is then evaluated using the same evaluation pipeline for a fair comparison with the baseline model.

In [46]:
# ============================================================
# Tune Random Forest
# ============================================================

rf_best_model, rf_tuned_results, rf_best_params = tune_model(
    model=RandomForestRegressor(random_state=42),
    param_distributions=random_forest_params,
    model_name="Random Forest",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    n_iter=20,
    cv=5
)


Tuning Random Forest...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)

This is taking too long, we give up.


Best Parameters:
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}

Best Cross Validation Score: 0.8435
Random Forest (Tuned) trained successfully!

Model                 : Random Forest (Tuned)
R2 Score              : 0.8669
CV Mean               : 0.8439
CV Std                : 0.0256
MAE                   : 18045.1480
MSE                   : 1020768896.2145
RMSE                  : 31949.4741
Training Time (s)     : 0.8064
Prediction Time (s)   : 0.0284


## 15.3 Hyperparameter Optimization - Gradient Boosting

Gradient Boosting achieved the highest baseline predictive performance among all evaluated regression models.

RandomizedSearchCV is applied to optimize the boosting parameters, including the number of estimators, learning rate, maximum tree depth, and subsampling ratio.

The optimized model is then evaluated using the unified evaluation framework for comparison with the baseline Gradient Boosting model.

In [47]:
# ============================================================
# Tune Gradient Boosting
# ============================================================

gb_best_model, gb_tuned_results, gb_best_params = tune_model(
    model=GradientBoostingRegressor(random_state=42),
    param_distributions=gradient_boosting_params,
    model_name="Gradient Boosting",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    n_iter=20,
    cv=5
)


Tuning Gradient Boosting...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters:
{'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05}

Best Cross Validation Score: 0.8640
Gradient Boosting (Tuned) trained successfully!

Model                 : Gradient Boosting (Tuned)
R2 Score              : 0.9140
CV Mean               : 0.8761
CV Std                : 0.0448
MAE                   : 15737.3528
MSE                   : 659500022.1101
RMSE                  : 25680.7325
Training Time (s)     : 4.2153
Prediction Time (s)   : 0.0188


## 15.4 Hyperparameter Optimization - Extra Trees

Extra Trees is another high-performing ensemble algorithm that introduces additional randomness during tree construction, often improving robustness while reducing variance.

RandomizedSearchCV is used to determine the optimal hyperparameter configuration. The tuned model is evaluated using the same standardized evaluation pipeline, enabling a fair comparison with both its baseline version and the other optimized ensemble models.

In [48]:
# ============================================================
# Tune Extra Trees
# ============================================================

et_best_model, et_tuned_results, et_best_params = tune_model(
    model=ExtraTreesRegressor(random_state=42),
    param_distributions=extra_trees_params,
    model_name="Extra Trees",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    n_iter=20,
    cv=5
)


Tuning Extra Trees...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters:
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}

Best Cross Validation Score: 0.8343
Extra Trees (Tuned) trained successfully!

Model                 : Extra Trees (Tuned)
R2 Score              : 0.8446
CV Mean               : 0.8322
CV Std                : 0.0104
MAE                   : 19235.3050
MSE                   : 1191829505.7352
RMSE                  : 34522.8838
Training Time (s)     : 0.7407
Prediction Time (s)   : 0.0391


# 16. Best Model Selection

After evaluating both the baseline and optimized regression models, the final model is selected based on predictive performance.

All evaluated models are ranked using the test-set R² score. The model with the highest predictive accuracy is selected as the final model for deployment and persistence.

This automated selection process ensures that the final choice is based on quantitative evaluation rather than manual inspection.

In [49]:
# ============================================================
# Collect All Model Results
# ============================================================

all_results = [
    linear_regression_results,
    ridge_results,
    lasso_results,
    decision_tree_results,
    random_forest_results,
    gradient_boosting_results,
    extra_trees_results,
    ada_boost_results,
    rf_tuned_results,
    gb_tuned_results,
    et_tuned_results
]

best_model_result = max(
    all_results,
    key=lambda x: x["R2 Score"]
)

best_model_name = best_model_result["Model"]
best_model = best_model_result["Model Object"]

print("=" * 70)
print("BEST MODEL SELECTED")
print("=" * 70)

print(f"Model : {best_model_name}")
print(f"R² Score : {best_model_result['R2 Score']:.4f}")
print(f"CV Mean : {best_model_result['CV Mean']:.4f}")
print(f"CV Std : {best_model_result['CV Std']:.4f}")


BEST MODEL SELECTED
Model : Gradient Boosting (Tuned)
R² Score : 0.9140
CV Mean : 0.8761
CV Std : 0.0448


# 17. Comprehensive Model Performance Comparison

A comprehensive comparison is performed across all baseline and optimized regression models.

Each model is evaluated using the same standardized evaluation framework, ensuring a fair comparison based on predictive accuracy, cross-validation performance, regression error metrics, and computational efficiency.

The models are ranked according to their test-set R² score, allowing clear identification of the strongest predictive model.

In [50]:
# ============================================================
# Comprehensive Model Comparison
# ============================================================

comparison_df = pd.DataFrame(all_results)

# Remove model objects before displaying/saving
comparison_df = comparison_df.drop(columns=["Model Object"])

# Sort by predictive performance
comparison_df = comparison_df.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

# Add ranking column
comparison_df.insert(
    0,
    "Rank",
    range(1, len(comparison_df) + 1)
)

# ============================================================
# Add Verdict Column
# ============================================================

# ============================================================
# Add Model Type Column
# ============================================================

def assign_model_type(model):

    if model in ["Linear Regression", "Ridge Regression", "Lasso Regression"]:
        return "Linear"

    elif model == "Decision Tree":
        return "Tree-Based"

    elif model in ["Random Forest", "Random Forest (Tuned)",
                   "Extra Trees", "Extra Trees (Tuned)"]:
        return "Bagging Ensemble"

    elif model in ["Gradient Boosting", "Gradient Boosting (Tuned)",
                   "AdaBoost"]:
        return "Boosting Ensemble"

    else:
        return "Other"

comparison_df["Model Type"] = comparison_df["Model"].apply(assign_model_type)


# ============================================================
# Add Verdict Column
# ============================================================

def assign_verdict(row):

    model = row["Model"]

    if row["Rank"] == 1:
        return "🏆 Final Selected Model"

    elif model == "Gradient Boosting":
        return "Excellent Baseline"

    elif model in ["Random Forest", "Extra Trees"]:
        return "Strong Baseline"

    elif model == "AdaBoost":
        return "Moderate Performance"

    elif model == "Decision Tree":
        return "High Variance"

    elif model in ["Ridge Regression",
                   "Lasso Regression",
                   "Linear Regression"]:
        return "Weak Baseline"

    elif model in ["Random Forest (Tuned)",
                   "Extra Trees (Tuned)"]:
        return "Tuning Reduced Performance"

    else:
        return "Evaluated"

comparison_df["Verdict"] = comparison_df.apply(assign_verdict, axis=1)

comparison_df


,Rank,Model,R2 Score,CV Mean,CV Std,MAE,MSE,RMSE,Training Time (s),Prediction Time (s),Model Type,Verdict
0,1,Gradient Boosting (Tuned),0.914019,0.876120,0.044787,15737.352811,6.595000e+08,25680.732507,4.215316,0.018823,Boosting Ensemble,🏆 Final Selected Model
1,2,Gradient Boosting,0.908392,0.879103,0.046479,16081.301901,7.026638e+08,26507.806187,2.933598,0.005505,Boosting Ensemble,Excellent Baseline
2,3,Random Forest,0.893977,0.845609,0.040231,17261.063606,8.132314e+08,28517.212528,1.663332,0.077920,Bagging Ensemble,Strong Baseline
3,4,Extra Trees,0.892548,0.848967,0.034512,17263.909030,8.241922e+08,28708.748487,2.835506,0.107671,Bagging Ensemble,Strong Baseline
4,5,Random Forest (Tuned),0.866920,0.843867,0.025602,18045.147984,1.020769e+09,31949.474115,0.806363,0.028410,Bagging Ensemble,Tuning Reduced Performance
5,6,AdaBoost,0.850289,0.776442,0.037556,22892.548798,1.148329e+09,33887.008303,5.718314,0.072028,Boosting Ensemble,Moderate Performance
6,7,Extra Trees (Tuned),0.844618,0.832206,0.010415,19235.305027,1.191830e+09,34522.883798,0.740669,0.039091,Bagging Ensemble,Tuning Reduced Performance
7,8,Decision Tree,0.802995,0.678294,0.103317,25484.365061,1.511094e+09,38872.791061,0.051317,0.004294,Tree-Based,High Variance
8,9,Ridge Regression,0.689416,0.784338,0.081473,21773.103759,2.382284e+09,48808.644478,0.023587,0.005470,Linear,Weak Baseline
9,10,Lasso Regression,0.654866,0.781781,0.079666,22116.840626,2.647293e+09,51451.851050,2.271728,0.002003,Linear,Weak Baseline


# 18. Save the Best Model

The highest-performing regression model identified during evaluation is persisted for deployment and future inference.

The model is serialized using Joblib, enabling efficient loading without retraining. The filename is generated dynamically based on the selected model to improve maintainability and reproducibility.

In [51]:
# ============================================================
# Save Best Model
# ============================================================

# Create models directory if it does not exist
os.makedirs("../models", exist_ok=True)

# Generate a safe filename from the selected model
safe_model_name = (
    best_model_name.lower()
    .replace(" ", "_")
    .replace("(", "")
    .replace(")", "")
)

best_model_path = f"../models/{safe_model_name}.pkl"

# Save model
joblib.dump(best_model, best_model_path)

print("=" * 70)
print("Best model saved successfully!")
print("=" * 70)
print(f"Selected Model : {best_model_name}")
print(f"Saved Location : {best_model_path}")


Best model saved successfully!
Selected Model : Gradient Boosting (Tuned)
Saved Location : ../models/gradient_boosting_tuned.pkl


# 19. Save Model Comparison Results

The comprehensive comparison of all evaluated regression models is exported as a CSV file.

This file provides a complete record of the experimental results and can be reused for reporting, visualization, and future benchmarking.

In [52]:
# ============================================================
# Save Model Comparison
# ============================================================

comparison_path = "../models/model_comparison.csv"

comparison_df.to_csv(comparison_path, index=False)

print("=" * 70)
print("Model comparison saved successfully!")
print("=" * 70)
print(f"Saved Location : {comparison_path}")
print(f"Total Models Compared : {len(comparison_df)}")


Model comparison saved successfully!
Saved Location : ../models/model_comparison.csv
Total Models Compared : 11
